In [2]:
from faster_whisper import WhisperModel

/home/aimore/GIT/transcription/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# model_size = "large-v3"
model_size = "medium"

#? Run on GPU with FP16
# model = WhisperModel(model_size, device="cuda", compute_type="float16")

#? or run on GPU with INT8
# model = WhisperModel(model_size, device="cuda", compute_type="int8_float16")

#? or run on CPU with INT8
model = WhisperModel(model_size, device="cpu", compute_type="int8")

In [ ]:
# file_path = "../data/audio_samples/sample.mp3"
file_path = "../data/audio_samples/sample2.wav"
from pydub import AudioSegment
import numpy as np

# Assuming `audio_segment` is your PyDub AudioSegment


audio_segment = AudioSegment.from_file(file_path, format="wav")
# audio_segment = np.array(audio_segment.get_array_of_samples())
# convert to expected format
print(audio_segment.frame_rate)
print(audio_segment.sample_width)
print(audio_segment.channels)

if audio_segment.frame_rate != 16000: # 16 kHz
    audio_segment = audio_segment.set_frame_rate(16000)
if audio_segment.sample_width != 2:   # int16
    audio_segment = audio_segment.set_sample_width(2)
if audio_segment.channels != 1:       # mono
    audio_segment = audio_segment.set_channels(1)        
arr = np.array(audio_segment.get_array_of_samples())
arr = arr.astype(np.float32)/32768.0


segments, info = model.transcribe(arr, beam_size=5)
a = "".join([x.text for x in segments])
a

In [ ]:
a = "".join([x.text for x in segments])
a

In [ ]:
print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

In [ ]:
# import whisperx
# import gc 

# device = "cuda" 
# audio_file = "audio.mp3"
# batch_size = 16 # reduce if low on GPU mem
# compute_type = "float16" # change to "int8" if low on GPU mem (may reduce accuracy)

# # 1. Transcribe with original whisper (batched)
# model = whisperx.load_model("large-v2", device, compute_type=compute_type)

# # save model to local path (optional)
# # model_dir = "/path/"
# # model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root=model_dir)

# audio = whisperx.load_audio(audio_file)
# result = model.transcribe(audio, batch_size=batch_size)
# print(result["segments"]) # before alignment

# # delete model if low on GPU resources
# # import gc; gc.collect(); torch.cuda.empty_cache(); del model

# # 2. Align whisper output
# model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
# result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

# print(result["segments"]) # after alignment

# # delete model if low on GPU resources
# # import gc; gc.collect(); torch.cuda.empty_cache(); del model_a

# # 3. Assign speaker labels
# diarize_model = whisperx.DiarizationPipeline(use_auth_token=YOUR_HF_TOKEN, device=device)

# # add min/max number of speakers if known
# diarize_segments = diarize_model(audio)
# # diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)

# result = whisperx.assign_word_speakers(diarize_segments, result)
# print(diarize_segments)
# print(result["segments"]) # segments are now assigned speaker IDs
